# TC3-82 – Levantar e integrar bases socioeconômicas e financeiras municipais

**Responsável:** Mauricio Cruz

**Sprint:** Fase 2

**Jira:** TC3-82

## Objetivo

Levantar, baixar, documentar e integrar as novas bases socioeconômicas e financeiras municipais que podem enriquecer as análises exploratórias e futuras etapas de modelagem.



### Carrega dados ml-alunos utilizando os passos do TC3_38

In [16]:
import polars as pl
import pandas as pd
import zipfile
from pathlib import Path

In [17]:

# Localiza a raiz do projeto a partir do diretório atual do notebook.
# Assim o notebook funciona no VS Code, Jupyter local ou outro clone do repositório.
def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dados" / "principal" / "ml_aluno.zip").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível encontrar dados/principal/ml_aluno.zip. "
        "Execute o notebook dentro do repositório ou confira se os dados foram baixados."
    )

PROJECT_ROOT = encontrar_raiz_projeto()
ZIP_PATH = PROJECT_ROOT / "dados" / "principal" / "ml_aluno.zip"
EXTRACT_PATH = PROJECT_ROOT / "dados" / "principal"

if not (EXTRACT_PATH / "ml_aluno").exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

# Caminho local da feature table extraída
BASE_PATH = EXTRACT_PATH / "ml_aluno"

# Partições da Gold
PATH_2023 = BASE_PATH / "ano2023"
PATH_2024 = BASE_PATH / "ano2024"

arquivos_2023 = sorted(PATH_2023.glob("*.parquet"))
arquivos_2024 = sorted(PATH_2024.glob("*.parquet"))

df_2023 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2023],
    ignore_index=True
)

df_2024 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2024],
    ignore_index=True
)

# Consolidação da base
df = pd.concat([df_2023, df_2024], ignore_index=True)



### Carrega arquivos complementares

In [18]:


PATH_COMPLEMENTAR = PROJECT_ROOT / "dados" / "complementar" 

#Bolsa Familia
df_bolsa_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_dados_bolsa_familia_2023_2024.csv")

#Fundeb
df_fundeb_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_dados_fundeb_indicador_municipal_2023_2024.csv")

#Receitas Orcamentarias
df_receitas_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_de_dados_municipio_receitas_orcamentarias_2023_2024.csv")

#PIB
df_pib_bronze = pd.read_csv(
    PATH_COMPLEMENTAR / "base_de_dados_PIB_Municipios_2022_2023.csv",
    sep=";",           # o arquivo usa ;
    decimal=","        # números com vírgula
)

#INSE
df_inse_bronze = pd.read_excel(PATH_COMPLEMENTAR / "INSE_2023_escolas.xlsx", sheet_name="INSE_ESC_2023")  # ou sheet_name=0 (primeira aba)


#Censo Escolar
ZIP_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.zip"
CSV_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.csv"

if ZIP_PATH_CENSO.exists() and not CSV_PATH_CENSO.exists():
    with zipfile.ZipFile(ZIP_PATH_CENSO, "r") as zip_ref:
        zip_ref.extractall(PATH_COMPLEMENTAR)

if CSV_PATH_CENSO.exists():
    df_censo_bronze = pd.read_csv(
        CSV_PATH_CENSO,
        sep=";",
        decimal=",",
        encoding="latin1"
    )
else:
    print("Arquivo do censo não encontrado.")








C:\Users\Mauripi\AppData\Local\Temp\ipykernel_13388\3810150610.py:32: DtypeWarning: Columns (31: CO_ORGAO_REGIONAL) have mixed types. Specify dtype option on import or set low_memory=False.
  df_censo_bronze = pd.read_csv(


### Padronizar os nomes das colunas

In [26]:
# Bolsa Família
df_bolsa_silver = (
    df_bolsa_bronze
    .rename(columns={
        "ano_referencia": "ano",
        "total_valor": "valor_bolsa_familia"
    })
    .rename(columns=lambda col: col.lower())
)

# Fundeb
df_fundeb_silver = (
    df_fundeb_bronze
    .rename(columns={
        "total_valor": "valor_fundeb"
    })
    .rename(columns=lambda col: col.lower())
)

# Receitas orçamentárias
df_receitas_silver = (
    df_receitas_bronze
    .rename(columns={
        "total_valor": "valor_receitas"
    })
    .rename(columns=lambda col: col.lower())
)

# PIB
df_pib_silver = (
    df_pib_bronze
    .rename(columns={
        "Ano": "ano",
        "UF": "sigla_uf",
        "Código do Município": "id_municipio",
        "Produto Interno Bruto, a preços correntes": "valor_pib",
        "Produto Interno Bruto per capita, a preços correntes": "valor_pib_per_capita"
    })
    .rename(columns=lambda col: col.lower())
)

# INSE
df_inse_silver = (
    df_inse_bronze
    .rename(columns={
        "NU_ANO_SAEB": "ano",
        "SG_UF": "sigla_uf",
        "CO_MUNICIPIO": "id_municipio",
    })
    .rename(columns=lambda col: col.lower())
)

# Censo
df_censo_silver = (
    df_censo_bronze
    .rename(columns={
        "NU_ANO_CENSO": "ano",
        "SG_UF": "sigla_uf",
        "CO_MUNICIPIO": "id_municipio",
    })
    .rename(columns=lambda col: col.lower())
)


### Padronizar os tipos das colunas

In [27]:
df["id_municipio"] = df["id_municipio"].astype("int64")
df["ano_referencia"] = df["ano_referencia"].astype("int64")

df_bolsa_silver["valor_bolsa_familia"] = df_bolsa_silver["valor_bolsa_familia"].astype("float64")
df_fundeb_silver["valor_fundeb"] = df_fundeb_silver["valor_fundeb"].astype("float64")
df_receitas_silver["valor_receitas"] = df_receitas_silver["valor_receitas"].astype("float64")
df_pib_silver["valor_pib"] = (
    df_pib_silver["valor_pib"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)
df_pib_silver["valor_pib_per_capita"] = (
    df_pib_silver["valor_pib_per_capita"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

colunas_com_codigo = [
    coluna
    for coluna in df_censo_silver.columns
    if df_censo_silver[coluna].astype("string").eq("0MI11").any()
]

for coluna in colunas_com_codigo:
    df_censo_silver[coluna] = df_censo_silver[coluna].astype("string")

### Unir todas as bases

#### Todas as fontes são consolidadas no nível município/ano antes do `join`. INSE e Censo chegam no nível escolar e são agregados para proteger a granularidade adotada e impedir a explosão de linhas.

In [29]:

df_aluno_silver = pl.DataFrame(df)
df_bolsa_silver = pl.DataFrame(df_bolsa_silver)  # Converte de Pandas para Polars
df_fundeb_silver = pl.DataFrame(df_fundeb_silver)  # Converte de Pandas para Polars
df_receitas_silver = pl.DataFrame(df_receitas_silver)  # Converte de Pandas para Polars
df_pib_silver = pl.DataFrame(df_pib_silver)  # Converte de Pandas para Polars
df_inse_silver = pl.DataFrame(df_inse_silver)  # Converte de Pandas para Polars
df_censo_silver = pl.DataFrame(df_censo_silver)  # Converte de Pandas para Polars 

chaves_municipio = ["ano", "id_municipio", "sigla_uf"]

# O INSE é escolar na origem. A média e os percentuais são ponderados
# pela quantidade de alunos representados em cada escola.
colunas_inse_ponderadas = [
    "media_inse",
    "pc_nivel_1", "pc_nivel_2", "pc_nivel_3", "pc_nivel_4",
    "pc_nivel_5", "pc_nivel_6", "pc_nivel_7", "pc_nivel_8",
]
df_inse_municipal = (
    df_inse_silver
    .with_columns(
        pl.col("qtd_alunos_inse").cast(pl.Float64, strict=False),
        *[pl.col(col).cast(pl.Float64, strict=False) for col in colunas_inse_ponderadas],
    )
    .group_by(chaves_municipio)
    .agg(
        pl.col("id_escola").n_unique().alias("qtd_escolas_inse"),
        pl.col("qtd_alunos_inse").sum().alias("qtd_alunos_inse"),
        *[
            (
                (pl.col(col) * pl.col("qtd_alunos_inse")).sum()
                / pl.col("qtd_alunos_inse").sum()
            ).alias(col)
            for col in colunas_inse_ponderadas
        ],
    )
)

# O Censo também é escolar na origem. Totais são somados e indicadores
# binários viram a proporção de escolas do município com cada recurso.
colunas_censo_totais = [
    "qt_mat_bas", "qt_doc_bas", "qt_tur_bas"
]
colunas_censo_indicadores = [
    "in_biblioteca_sala_leitura", "in_laboratorio_informatica", "in_quadra_esportes",
    "in_agua_rede_publica", "in_energia_rede_publica",
    "in_esgoto_rede_publica", "in_internet", "in_banda_larga",
]
df_censo_municipal = (
    df_censo_silver
    .with_columns(
        *[
            pl.col(col).cast(pl.Float64, strict=False)
            for col in colunas_censo_totais + colunas_censo_indicadores
        ]
    )
    .group_by(chaves_municipio)
    .agg(
        pl.col("co_entidade").n_unique().alias("qtd_escolas_censo"),
        *[pl.col(col).sum().alias(col) for col in colunas_censo_totais],
        *[
            pl.col(col).mean().alias(col)
            for col in colunas_censo_indicadores
        ],
    )
)

qtd_linhas_antes_dos_joins = df_aluno_silver.height

# Junta Bolsa Família
df_aluno_silver = df_aluno_silver.join(
    df_bolsa_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

# Junta Fundeb
df_aluno_silver = df_aluno_silver.join(
    df_fundeb_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

# Junta Receitas
df_aluno_silver = df_aluno_silver.join(
    df_receitas_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

df_aluno_silver = df_aluno_silver.join(
    df_pib_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    validate="m:1"
)

# Junta INSE já consolidado por município
df_aluno_silver = df_aluno_silver.join(
    df_inse_municipal,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=chaves_municipio,
    validate="m:1"
)

# Junta Censo Escolar já consolidado por município
df_aluno_silver = df_aluno_silver.join(
    df_censo_municipal,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=chaves_municipio,
    validate="m:1"
)

assert df_aluno_silver.height == qtd_linhas_antes_dos_joins, (
    "A quantidade de alunos mudou durante os joins: "
    f"{qtd_linhas_antes_dos_joins:,} -> {df_aluno_silver.height:,}"
)


In [30]:
print(f"Shape: {df_aluno_silver.shape}")
df_aluno_silver.glimpse()
df_aluno_silver.head()

Shape: (3867999, 49)
Rows: 3867999
Columns: 49
$ id_aluno                                           <str> '31062133', '15078120', '43090743', '42097998', '43072338', '33048792', '24009445', '33014166', '11010561', '32035154'
$ id_escola                                          <str> '60021550', '60001463', '60037407', '60035930', '60037775', '60024671', '60010376', '60023928', '60000036', '60022642'
$ id_municipio                                       <i64> 3136900, 1506005, 4305108, 4219507, 4309209, 3302700, 2401305, 3300704, 1100205, 3204708
$ nome_municipio                                     <str> 'Juruaia', 'Prainha', 'Caxias do Sul', 'Xanxerê', 'Gravataí', 'Maricá', 'Campo Grande', 'Cabo Frio', 'Porto Velho', 'São Gabriel da Palha'
$ sigla_uf                                           <str> 'MG', 'PA', 'RS', 'SC', 'RS', 'RJ', 'RN', 'RJ', 'RO', 'ES'
$ ano_referencia                                     <i64> 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023
$ serie        

id_aluno,id_escola,id_municipio,nome_municipio,sigla_uf,ano_referencia,serie,rede,rede_descricao,caderno,presenca,preenchimento_caderno,proficiencia,peso_aluno,taxa_alfabetizacao_municipio_ano_anterior,media_portugues_municipio_ano_anterior,dataset_split,alfabetizado,processed_at,valor_bolsa_familia,valor_fundeb,valor_receitas,nome uf,nome do município,valor_pib,valor_pib_per_capita,qtd_escolas_inse,qtd_alunos_inse,media_inse,pc_nivel_1,pc_nivel_2,pc_nivel_3,pc_nivel_4,pc_nivel_5,pc_nivel_6,pc_nivel_7,pc_nivel_8,qtd_escolas_censo,qt_mat_bas,qt_doc_bas,qt_tur_bas,in_biblioteca_sala_leitura,in_laboratorio_informatica,in_quadra_esportes,in_agua_rede_publica,in_energia_rede_publica,in_esgoto_rede_publica,in_internet,in_banda_larga
str,str,i64,str,str,i64,i32,i32,str,str,i32,i32,f64,f64,f64,f64,str,i32,datetime[ns],f64,f64,f64,str,str,f64,f64,u64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""31062133""","""60021550""",3136900,"""Juruaia""","""MG""",2023,2,3,"""Municipal""","""6""",1,1,731.2356,1.322718,null,null,"""treino""",0,2026-09-01 18:03:53.794418,2.312976e6,4.0045e7,4.9867e8,"""Minas Gerais""","""Juruaia""",3.03328157e8,27366.31,4,365.0,5.18189,0.0,3.552137,13.774877,21.857534,27.463945,22.103781,11.241507,0.0,7,2143.0,114.0,104.0,0.428571,0.142857,0.571429,0.857143,1.0,0.714286,1.0,1.0
"""15078120""","""60001463""",1506005,"""Prainha""","""PA""",2023,2,3,"""Municipal""","""6""",1,1,732.783236,1.3249731,null,null,"""teste""",0,2026-09-01 18:03:53.794418,5.7458393e7,2.2302e8,null,"""Pará""","""Prainha""",5.48886134e8,15428.12,25,1556.0,3.887114,15.029878,40.511697,23.331555,12.725039,5.014814,2.360617,1.030463,0.0,124,12179.0,597.0,658.0,0.09434,0.009434,0.075472,0.150943,0.509434,0.018868,0.367925,0.615385
"""43090743""","""60037407""",4305108,"""Caxias do Sul""","""RS""",2023,2,3,"""Municipal""","""6""",1,1,755.341541,1.3818684,null,null,"""treino""",1,2026-09-01 18:03:53.794418,8.7984651e7,1.4014e9,2.5271e10,"""Rio Grande do Sul""","""Caxias do Sul""",3.7861e10,81683.97,124,13823.0,5.524075,0.061658,1.770355,6.163758,14.475369,24.831173,27.450799,23.430008,1.816964,420,94962.0,5877.0,4749.0,0.68642,0.385185,0.338272,0.995062,0.997531,0.958025,0.997531,0.94802
"""42097998""","""60035930""",4219507,"""Xanxerê""","""SC""",2023,2,3,"""Municipal""","""6""",1,1,711.643712,2.0121879,null,null,"""treino""",0,2026-09-01 18:03:53.794418,6.345159e6,2.1612e8,2.1373e9,"""Santa Catarina""","""Xanxerê""",3.1456e9,60952.29,20,1584.0,5.492967,0.0,1.978005,6.863251,15.859703,24.422923,25.647096,23.055972,2.174122,53,11783.0,881.0,649.0,0.596154,0.423077,0.442308,0.884615,1.0,0.326923,1.0,0.980769
"""43072338""","""60037775""",4309209,"""Gravataí""","""RS""",2023,2,3,"""Municipal""","""7""",0,0,null,null,null,null,"""treino""",0,2026-09-01 18:03:53.794418,1.10156423e8,7.1742e8,1.0518e10,"""Rio Grande do Sul""","""Gravataí""",1.5546e10,58646.11,85,8578.0,5.401837,0.244155,2.778186,7.99628,17.907062,26.069724,23.591075,19.782966,1.631288,185,54762.0,3581.0,2724.0,0.631285,0.24581,0.340782,0.944134,0.994413,0.759777,0.994413,0.94382


### Salvar a base consolidada

O dataset abaixo mantém uma linha por aluno, reúne as variáveis municipais disponíveis para o respectivo ano e grava os arquivos Parquet em partições por UF.

In [31]:
OUTPUT_DIR = (
    PROJECT_ROOT / "dados" / "consolidado" / "base_alunos_municipios_por_uf"
)

df_aluno_silver.write_parquet(
    OUTPUT_DIR,
    compression="zstd",
    statistics=True,
    partition_by="sigla_uf",
    mkdir=True,
)

arquivos_parquet = list(OUTPUT_DIR.rglob("*.parquet"))
tamanho_mb = sum(arq.stat().st_size for arq in arquivos_parquet) / (1024 ** 2)
print(f"Base consolidada particionada salva em: {OUTPUT_DIR}")
print(f"Linhas: {df_aluno_silver.height:,} | Colunas: {df_aluno_silver.width:,}")
print(f"Partições/arquivos: {len(arquivos_parquet)}")
print(f"Tamanho: {tamanho_mb:,.2f} MB")

Base consolidada particionada salva em: C:\Users\Mauripi\Desktop\tech_challenge_3\dados\consolidado\base_alunos_municipios_por_uf
Linhas: 3,867,999 | Colunas: 49
Partições/arquivos: 26
Tamanho: 131.93 MB
